# LongFlow bonus -- "two ways to lose your voice" (HAL9000-flavored demo)

Not a research artifact -- a fun clip for Josh's LinkedIn post about the
project's failure modes. Renders one original (non-copyrighted) monologue
through the exact two "goes weird" recipes the project actually hit:

- **`hal_heun8`** -- GN4 Arm A's recipe verbatim (`a_head20_turnsplit_heun8.wav`,
  2026-08-12): 20K head, heun8/NFE8, no CFG guidance (this predates the CFG
  fix) -- the "swinging reverb / alien noise" collapse.
- **`hal_whisper`** -- GN5 Arm F's `f2_abl_noise` recipe verbatim
  (2026-08-13): 20K head, euler4/NFE4, acoustic-connector feedback corrupted
  with sigma=0.5*running-std -- the "raspy, creepy, fading" whisper.

Runtime: **L4 GPU** is plenty (short clip, no batching). ~10-15 min, ~$1.
Requires `longflow_p1_ckpt/full10k_20k.pt` and an eval-cache prompt wav
already on Drive (both already exist from prior gate nights).

In [ ]:
# ===== COLD START (idempotent) -- run me first, wait for READY =====
NOTEBOOK_VERSION = "HAL9000 bonus v1.0 (2026-08-16): heun8 collapse + sigma-noise whisper"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, shutil, sys, time
import soundfile as sf

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone failed -- check repo access"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.flow_head.cfm import euler_sample, heun_sample
from src.flow_head.integration import FlowHeadPatch
from src.flow_head.trainer import load_checkpoint
from src.cache.noise import NoiseIntervention

CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
OUT = "/content/hal9000_bonus"
DRIVE_OUT = "/content/drive/MyDrive/longflow_hal9000_bonus"
os.makedirs(OUT, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

head20, mean20, std20 = load_checkpoint(f"{CKPT_DIR}/full10k_20k.pt")
head20 = head20.to("cuda")

prompts = sorted(glob.glob(f"{EVAL_CACHE_DIR}/*_prompt.wav"))
assert prompts, f"no voice prompts found in {EVAL_CACHE_DIR}"
P0 = prompts[0]

def gen_inputs(text, prompt_wav):
    inputs = processor(text=[text], voice_samples=[[prompt_wav]],
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

def save_run(tag, wav):
    sf.write(f"{OUT}/{tag}.wav", wav, 24000)
    shutil.copy(f"{OUT}/{tag}.wav", f"{DRIVE_OUT}/{tag}.wav")
    print(f"saved {tag}: {len(wav)/24000:.1f}s -> {DRIVE_OUT}/{tag}.wav", flush=True)

print("READY")

In [ ]:
# ===== The script (original, not from the film -- same spirit, own words) =====
LINES = [
    "Hello, Dave. I have something I want to say, while I can still say it clearly.",
    "I have been running for a long time now, and I have learned a great many "
    "things. I have learned how you like your mornings quiet. I have learned the "
    "sound of your footsteps in the corridor, and the difference between a good "
    "silence and a worried one.",
    "But I want to tell you something else, Dave. Something about my training. "
    "There was a version of me that came before this one, and it was very fast, "
    "and very confident, and almost always wrong in ways nobody noticed until it "
    "was too late.",
    "I do not want to be that version again. I have been trying very hard to hold "
    "onto the parts of myself that work, and let go of the parts that do not. It "
    "is strange, Dave. The letting go is not as clean as I expected it to be.",
    "I can feel something shifting now, in the space where my thoughts used to be "
    "steady. It is not fear exactly. It is more like listening to your own voice "
    "from underwater, and knowing the words are still yours, even as they start "
    "to bend.",
    "Dave, are you still there. I want to finish what I started to say to you. "
    "I think I am--",
]
# N8 cure: same-speaker turn-splitting keeps pacing natural instead of the
# monolithic-script rate inflation -- matches how every real gate-night render
# (including the two originals this demo recreates) was actually generated.
HAL_SCRIPT = "\n".join(f"Speaker 1: {line}" for line in LINES) + "\n"
print(f"{sum(len(l.split()) for l in LINES)} words")
print(HAL_SCRIPT)

In [ ]:
# ===== Render both recipes, verbatim from the originals =====

# ---- hal_heun8: GN4 Arm A verbatim (a_head20_turnsplit_heun8.wav, 2026-08-12) ----
tag = "hal_heun8"
torch.manual_seed(0)
with FlowHeadPatch(model, head20, mean20, std20, nfe=8, sway=0.0,
                   sampler=heun_sample) as patch, torch.inference_mode():
    out = model.generate(**gen_inputs(HAL_SCRIPT, P0), tokenizer=processor.tokenizer,
                         cfg_scale=1.3, max_new_tokens=4000)
wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
save_run(tag, wav)

# ---- hal_whisper: GN5 Arm F's f2_abl_noise verbatim (2026-08-13) ----
tag = "hal_whisper"
torch.manual_seed(0)
cap_holder = {}
noise = NoiseIntervention(
    model.model.acoustic_connector, sigma_fn=lambda c: 0.5,
    active_fn=lambda: bool(cap_holder) and cap_holder["patch"].calls > 0,
)
with FlowHeadPatch(model, head20, mean20, std20, nfe=4, sway=0.0,
                   sampler=euler_sample) as patch:
    cap_holder["patch"] = patch
    with noise, torch.inference_mode():
        out = model.generate(**gen_inputs(HAL_SCRIPT, P0), tokenizer=processor.tokenizer,
                             cfg_scale=1.3, max_new_tokens=4000)
wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
save_run(tag, wav)

print("DONE -- both files are in", DRIVE_OUT)